# Advanced Problems with Solutions: Rounding

Python version target: **Python 3.13**

Topics covered:
- `round(number)` vs `round(number, ndigits)`
- Banker's rounding / ties to even
- Positive, zero, and negative `ndigits`
- Floating-point surprises
- Custom half-away-from-zero rounding
- `Decimal` rounding modes
- Robust testing and real-world use cases

## Setup

In [1]:
from decimal import Decimal, ROUND_HALF_EVEN, ROUND_HALF_UP, ROUND_HALF_DOWN, ROUND_UP, ROUND_DOWN
from fractions import Fraction
from math import copysign, floor, ceil, isclose
import math

## Problem 1: Return Type Trap

Predict the value and type of each expression:

```python
round(1.5)
round(1.5, 0)
round(1.5, 1)
round(10)
round(10, 0)
```

In [2]:
expressions = [
    round(1.5),
    round(1.5, 0),
    round(1.5, 1),
    round(10),
    round(10, 0),
]

[(value, type(value).__name__) for value in expressions]

[(2, 'int'), (2.0, 'float'), (1.5, 'float'), (10, 'int'), (10, 'int')]

### Solution

When `round()` is called with one argument, the result is an `int`.

When `ndigits` is supplied, even if it is `0`, the result usually keeps the same numeric type as the input.

So `round(1.5)` gives `2`, but `round(1.5, 0)` gives `2.0`.

## Problem 2: Fix the Notebook Bug

The following code contains a mistake:

```python
a = round(1.5, 0)
a, type(b)
```

Fix it and explain the output.

In [3]:
a = round(1.5, 0)
a, type(a)

(2.0, float)

### Solution

`b` was never defined. The correct variable is `a`.

`round(1.5, 0)` returns `2.0`, which is a `float`, because the `ndigits` argument was supplied.

## Problem 3: Banker's Rounding Table

Create a table showing how Python rounds these values:

```python
[0.5, 1.5, 2.5, 3.5, 4.5, -0.5, -1.5, -2.5, -3.5, -4.5]
```

Explain the pattern.

In [4]:
values = [0.5, 1.5, 2.5, 3.5, 4.5, -0.5, -1.5, -2.5, -3.5, -4.5]

table = [(x, round(x)) for x in values]
table

[(0.5, 0),
 (1.5, 2),
 (2.5, 2),
 (3.5, 4),
 (4.5, 4),
 (-0.5, 0),
 (-1.5, -2),
 (-2.5, -2),
 (-3.5, -4),
 (-4.5, -4)]

### Solution

Python uses rounding to nearest, with ties going to the nearest even integer.

Examples:
- `round(1.5)` becomes `2`
- `round(2.5)` also becomes `2`
- `round(3.5)` becomes `4`

This is often called banker's rounding.

## Problem 4: Negative `ndigits`

Predict the output:

```python
round(149, -1)
round(150, -1)
round(250, -2)
round(350, -2)
round(850, -2)
```

In [5]:
results = [
    round(149, -1),
    round(150, -1),
    round(250, -2),
    round(350, -2),
    round(850, -2),
]

results

[150, 150, 200, 400, 800]

### Solution

A negative `ndigits` rounds to tens, hundreds, thousands, and so on.

Ties still use the ties-to-even rule.

So `round(250, -2)` becomes `200`, because `200` is the even hundred multiple, while `round(350, -2)` becomes `400`.

## Problem 5: Floating-Point Surprise

Explain why this may surprise you:

```python
round(2.675, 2)
```

Then solve it using `Decimal`.

In [6]:
round(2.675, 2)

2.67

In [7]:
Decimal("2.675").quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

Decimal('2.68')

### Solution

`2.675` cannot be represented exactly as a binary floating-point number.

The stored value is slightly different from the decimal value you typed.

Use `Decimal` with string input when decimal precision matters.

## Problem 6: Implement Half Away From Zero

Write a function `round_half_away_from_zero(x)` that rounds to the nearest integer, with exact halves moving away from zero.

Examples:
- `1.5 -> 2`
- `2.5 -> 3`
- `-1.5 -> -2`
- `-2.5 -> -3`

In [8]:
def round_half_away_from_zero(x: float) -> int:
    return int(x + copysign(0.5, x))


values = [1.4, 1.5, 2.5, -1.4, -1.5, -2.5, 0.5, -0.5]
[(x, round(x), round_half_away_from_zero(x)) for x in values]

[(1.4, 1, 1),
 (1.5, 2, 2),
 (2.5, 2, 3),
 (-1.4, -1, -1),
 (-1.5, -2, -2),
 (-2.5, -2, -3),
 (0.5, 0, 1),
 (-0.5, 0, -1)]

### Solution

For positive numbers, add `0.5` and truncate.

For negative numbers, subtract `0.5` and truncate.

`copysign(0.5, x)` gives `0.5` for positive `x` and `-0.5` for negative `x`.

## Problem 7: General Half Away From Zero with Decimal Places

Extend the previous function so it supports `ndigits`.

Example:

```python
round_half_away_from_zero(1.25, 1) == 1.3
round_half_away_from_zero(-1.25, 1) == -1.3
```

In [9]:
def round_half_away_from_zero_ndigits(x: float, ndigits: int = 0):
    scale = 10 ** ndigits
    shifted = x * scale
    rounded = int(shifted + copysign(0.5, shifted))
    result = rounded / scale
    return int(result) if ndigits == 0 else result


tests = [
    (1.25, 1),
    (1.35, 1),
    (-1.25, 1),
    (-1.35, 1),
    (125, -1),
    (-125, -1),
]

[(x, n, round(x, n), round_half_away_from_zero_ndigits(x, n)) for x, n in tests]

[(1.25, 1, 1.2, 1.3),
 (1.35, 1, 1.4, 1.4),
 (-1.25, 1, -1.2, -1.3),
 (-1.35, 1, -1.4, -1.4),
 (125, -1, 120, 130.0),
 (-125, -1, -120, -130.0)]

### Solution

Scaling lets us reuse integer rounding logic.

However, this version still uses binary floats, so it is not ideal for money or legally sensitive decimal calculations.

For exact decimal rounding, use `Decimal`.

## Problem 8: Decimal Half-Up Rounding

Use `Decimal` to implement traditional school-style rounding to a fixed number of decimal places.

Round these values to one decimal place:

```python
Decimal("1.25")
Decimal("1.35")
Decimal("-1.25")
Decimal("-1.35")
```

In [10]:
values = [Decimal("1.25"), Decimal("1.35"), Decimal("-1.25"), Decimal("-1.35")]
quantum = Decimal("0.1")

[(x, x.quantize(quantum, rounding=ROUND_HALF_UP)) for x in values]

[(Decimal('1.25'), Decimal('1.3')),
 (Decimal('1.35'), Decimal('1.4')),
 (Decimal('-1.25'), Decimal('-1.3')),
 (Decimal('-1.35'), Decimal('-1.4'))]

### Solution

`ROUND_HALF_UP` rounds ties away from zero for positive and negative values in this context.

`Decimal` is preferred when the decimal representation itself is meaningful, such as currencies, tax rates, or report values.

## Problem 9: Compare Rounding Modes

For each value below, compare:

- `ROUND_HALF_EVEN`
- `ROUND_HALF_UP`
- `ROUND_HALF_DOWN`
- `ROUND_UP`
- `ROUND_DOWN`

Use values with both positive and negative signs.

In [11]:
values = [Decimal("2.5"), Decimal("3.5"), Decimal("-2.5"), Decimal("-3.5")]
quantum = Decimal("1")

modes = {
    "HALF_EVEN": ROUND_HALF_EVEN,
    "HALF_UP": ROUND_HALF_UP,
    "HALF_DOWN": ROUND_HALF_DOWN,
    "UP": ROUND_UP,
    "DOWN": ROUND_DOWN,
}

[
    {"x": x, **{name: x.quantize(quantum, rounding=mode) for name, mode in modes.items()}}
    for x in values
]

[{'x': Decimal('2.5'),
  'HALF_EVEN': Decimal('2'),
  'HALF_UP': Decimal('3'),
  'HALF_DOWN': Decimal('2'),
  'UP': Decimal('3'),
  'DOWN': Decimal('2')},
 {'x': Decimal('3.5'),
  'HALF_EVEN': Decimal('4'),
  'HALF_UP': Decimal('4'),
  'HALF_DOWN': Decimal('3'),
  'UP': Decimal('4'),
  'DOWN': Decimal('3')},
 {'x': Decimal('-2.5'),
  'HALF_EVEN': Decimal('-2'),
  'HALF_UP': Decimal('-3'),
  'HALF_DOWN': Decimal('-2'),
  'UP': Decimal('-3'),
  'DOWN': Decimal('-2')},
 {'x': Decimal('-3.5'),
  'HALF_EVEN': Decimal('-4'),
  'HALF_UP': Decimal('-4'),
  'HALF_DOWN': Decimal('-3'),
  'UP': Decimal('-4'),
  'DOWN': Decimal('-3')}]

### Solution

`ROUND_HALF_EVEN` matches Python's built-in `round()` tie behavior.

`ROUND_UP` means away from zero.

`ROUND_DOWN` means toward zero.

The words `up` and `down` in `Decimal` do not mean positive infinity and negative infinity.

## Problem 10: Avoid Rounding Before Summing

A report rounds each transaction to two decimals before summing:

```python
sum(round(x, 2) for x in values)
```

Explain why this can produce a different result from summing first and rounding once.

In [12]:
values = [0.015, 0.015, 0.015]

round_each_then_sum = sum(round(x, 2) for x in values)
sum_then_round = round(sum(values), 2)

round_each_then_sum, sum_then_round

(0.03, 0.04)

### Solution

Rounding loses information.

If you round many small values before aggregation, the small errors can accumulate.

Best practice: keep full precision internally and round only for final display, unless the business rule explicitly requires per-item rounding.

## Problem 11: Significant Figures

Write a function `round_sig(x, sig)` that rounds a number to a given number of significant figures.

Examples:
- `round_sig(12345, 3) -> 12300`
- `round_sig(0.012345, 3) -> 0.0123`
- `round_sig(-98765, 2) -> -99000`

In [13]:
def round_sig(x: float, sig: int) -> float:
    if sig <= 0:
        raise ValueError("sig must be positive")
    if x == 0:
        return 0

    digits = sig - 1 - floor(math.log10(abs(x)))
    return round(x, digits)


tests = [12345, 0.012345, -98765, 1.2345, 999.9, 0]
[(x, round_sig(x, 3)) for x in tests]

[(12345, 12300),
 (0.012345, 0.0123),
 (-98765, -98800),
 (1.2345, 1.23),
 (999.9, 1000.0),
 (0, 0)]

### Solution

The key is to compute how many decimal places are needed based on the order of magnitude of `x`.

`log10(abs(x))` tells us the magnitude.

This implementation uses Python's normal `round()`, so ties still use banker's rounding.

## Problem 12: Rounding Money Safely

Compute a 7.5% tax on these prices and round to cents using `Decimal`:

```python
["19.99", "5.49", "100.00"]
```

Use `ROUND_HALF_UP`, which is common in financial contexts.

In [14]:
prices = [Decimal("19.99"), Decimal("5.49"), Decimal("100.00")]
tax_rate = Decimal("0.075")
cent = Decimal("0.01")

taxes = [
    (price, (price * tax_rate).quantize(cent, rounding=ROUND_HALF_UP))
    for price in prices
]

taxes

[(Decimal('19.99'), Decimal('1.50')),
 (Decimal('5.49'), Decimal('0.41')),
 (Decimal('100.00'), Decimal('7.50'))]

### Solution

Avoid binary floats for currency.

Use `Decimal` values created from strings.

Then use `quantize()` to round to cents.

## Problem 13: Rounding Fractions Exactly

Write a function that rounds a `Fraction` to the nearest integer using ties-to-even, matching Python's `round()` behavior.

In [15]:
def round_fraction_half_even(x: Fraction) -> int:
    lower = x.numerator // x.denominator
    remainder = x - lower

    if remainder < Fraction(1, 2):
        return lower
    if remainder > Fraction(1, 2):
        return lower + 1

    return lower if lower % 2 == 0 else lower + 1


tests = [Fraction(1, 2), Fraction(3, 2), Fraction(5, 2), Fraction(-1, 2), Fraction(-3, 2), Fraction(-5, 2)]
[(x, round(x), round_fraction_half_even(x)) for x in tests]

[(Fraction(1, 2), 0, 0),
 (Fraction(3, 2), 2, 2),
 (Fraction(5, 2), 2, 2),
 (Fraction(-1, 2), 0, 0),
 (Fraction(-3, 2), -2, -2),
 (Fraction(-5, 2), -2, -2)]

### Solution

`Fraction` allows exact rational arithmetic.

The only special case is when the fractional distance is exactly `1/2`.

At that point, choose the even integer.

## Problem 14: Custom Class with `__round__`

Create a class `Measurement` that stores a numeric value and supports `round(obj)` and `round(obj, ndigits)`.

In [16]:
class Measurement:
    def __init__(self, value: float, unit: str):
        self.value = float(value)
        self.unit = unit

    def __round__(self, ndigits=None):
        if ndigits is None:
            return Measurement(round(self.value), self.unit)
        return Measurement(round(self.value, ndigits), self.unit)

    def __repr__(self):
        return f"Measurement({self.value!r}, {self.unit!r})"


m = Measurement(12.3456, "cm")
round(m), round(m, 2)

(Measurement(12.0, 'cm'), Measurement(12.35, 'cm'))

### Solution

The built-in `round()` calls `obj.__round__()` for custom objects.

Supporting `ndigits=None` lets the class behave like built-in numeric types.

## Problem 15: Mini Project — Rounding Policy Engine

Write a function `round_values(values, ndigits, policy)` that supports these policies:

- `'half_even'`
- `'half_up'`
- `'down'`
- `'up'`

Use `Decimal` internally and accept values as strings.

In [17]:
ROUNDING_POLICIES = {
    "half_even": ROUND_HALF_EVEN,
    "half_up": ROUND_HALF_UP,
    "down": ROUND_DOWN,
    "up": ROUND_UP,
}


def round_values(values: list[str], ndigits: int, policy: str) -> list[Decimal]:
    if policy not in ROUNDING_POLICIES:
        raise ValueError(f"unknown policy: {policy!r}")

    quantum = Decimal("1").scaleb(-ndigits)
    rounding_mode = ROUNDING_POLICIES[policy]

    return [
        Decimal(value).quantize(quantum, rounding=rounding_mode)
        for value in values
    ]


values = ["1.25", "1.35", "-1.25", "-1.35"]
{policy: round_values(values, 1, policy) for policy in ROUNDING_POLICIES}

{'half_even': [Decimal('1.2'),
  Decimal('1.4'),
  Decimal('-1.2'),
  Decimal('-1.4')],
 'half_up': [Decimal('1.3'), Decimal('1.4'), Decimal('-1.3'), Decimal('-1.4')],
 'down': [Decimal('1.2'), Decimal('1.3'), Decimal('-1.2'), Decimal('-1.3')],
 'up': [Decimal('1.3'), Decimal('1.4'), Decimal('-1.3'), Decimal('-1.4')]}

### Solution

A policy engine makes rounding rules explicit.

This is better than scattering calls to `round()` throughout a codebase.

For business logic, the rounding policy should be named, documented, and tested.

## Final Best Practices Summary

- `round(x)` returns an `int` for built-in numeric inputs.
- `round(x, ndigits)` usually preserves the input's numeric type.
- Python uses ties-to-even rounding, not always school-style rounding.
- Negative `ndigits` rounds to tens, hundreds, thousands, and so on.
- Do not use binary floats for exact decimal requirements.
- Use `Decimal(str_value)` or `Decimal("...")`, not `Decimal(float_value)`.
- Use `quantize()` for fixed decimal places.
- Round late, not early, unless rules require per-item rounding.
- Make business rounding policies explicit and tested.